# Multi-Agent RAG — Cleaned Up (Hybrid: Local Retrieval/Rerank + OpenRouter Generation)
Deduplicated from your working notebook. Architecture that survived debugging:
- **Embeddings + BM25 + vector retrieval**: local (Ollama `nomic-embed-text`, Chroma, BM25) — no rate limits, already working
- **Reranking**: local CPU `CrossEncoder` (moved off GPU to avoid VRAM conflicts with the LLM)
- **Generation**: OpenRouter (`openai/gpt-oss-120b:free`) with retry logic — swapped in for reliability

**Fixed one bug from your notebook**: the model string was `together_ai/together/gpt-oss-120b`, which isn't a valid OpenRouter slug. Correct one is `openai/gpt-oss-120b:free`.


## 1. Cold start — Ollama (embeddings, BM25, reranker still need it running for embeddings)

In [ ]:
import subprocess, time, os

!apt-get update -qq && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!ollama pull nomic-embed-text
!ollama list


## 2. Install packages (single coordinated install — avoids the version-conflict spiral from before)

In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-ollama langgraph \
    llama-index llama-index-llms-ollama llama-index-embeddings-ollama llama-index-vector-stores-chroma \
    llama-index-readers-file llama-index-retrievers-bm25 \
    chromadb pypdf docx2txt sentence-transformers nest_asyncio tenacity


In [ ]:
import nest_asyncio
nest_asyncio.apply()

DOCS_DIR = '/content/Office_Docs'
PERSIST_DIR = '/content/chroma_store'
COLLECTION_NAME = 'office_policies_v2'


## 3. LlamaIndex settings — embeddings local via Ollama

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text", base_url="http://127.0.0.1:11434")
# Note: Settings.llm is NOT set to Ollama here anymore — generation goes through OpenRouter (Cell 8),
# not through LlamaIndex's query engine. Settings.llm only matters if you use index.as_query_engine()
# directly, which we don't in the final pipeline.


## 4. Load, chunk, embed into Chroma (skips re-embedding if already indexed)

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

documents = SimpleDirectoryReader(DOCS_DIR).load_data()
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64)
nodes = splitter.get_nodes_from_documents(documents)
print(f"Loaded {len(documents)} docs -> {len(nodes)} chunks")

chroma_client = chromadb.PersistentClient(path=PERSIST_DIR)
chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

if chroma_collection.count() == 0:
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(nodes, storage_context=storage_context)
    print(f"Indexed {chroma_collection.count()} chunks.")
else:
    index = VectorStoreIndex.from_vector_store(vector_store)
    print(f"Loaded existing {chroma_collection.count()} chunks — skipped re-embedding.")


## 5. Hybrid retrieval — vector + BM25 fusion (wide candidate pool, trimmed after fusion)

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever

vector_retriever = index.as_retriever(similarity_top_k=10)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=10)
fusion_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=10,   # keep wide here; reranker below trims to the real final 4
    num_queries=1,
    mode="reciprocal_rerank",
)


## 6. Reranker — local CPU CrossEncoder (final decision: GPU caused VRAM conflicts)

In [ ]:
import torch, gc
from sentence_transformers import CrossEncoder

def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    gc.collect()

clear_memory()
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu')
print("Reranker ready on CPU.")


## 7. Generation — OpenRouter (`openai/gpt-oss-120b:free`) with retry logic
**Fixed**: correct model slug is `openai/gpt-oss-120b:free`, not `together_ai/together/gpt-oss-120b`.
Needs `OPENROUTER_API_KEY` in Colab Secrets (key icon, left sidebar).


In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import openai

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

llm = ChatOpenAI(
    model="openai/gpt-oss-120b:free",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIStatusError, openai.APIConnectionError)),
    wait=wait_exponential(multiplier=2, min=4, max=60),
    stop=stop_after_attempt(5),
)
def call_llm(prompt: str) -> str:
    return llm.invoke(prompt).content

print(call_llm("Say hello in 5 words."))


## 8. Policy Q&A agent — final version
Retrieval (fusion) -> rerank (CrossEncoder) -> generation (OpenRouter, retry-wrapped).
This is the "complete, include everything, don't over-decline" prompt version from your debugging.


In [ ]:
POLICY_QA_PROMPT = """You are a company policy assistant. Answer using ONLY the context below.
Format your answer as a bulleted list, with one specific fact per bullet.

Rules:
- If the context fully answers the question, answer completely with every relevant detail (names, numbers, dates, vendors).
- If the context does NOT contain the specific answer, but DOES contain related information, say so and share that related info instead of just declining.
- Only say "I don't have that information in the company documents." if there is truly nothing relevant in the context.

Context:
{context}

Question: {question}

Answer:"""

async def policy_rag_agent(question: str) -> str:
    candidates = await fusion_retriever.aretrieve(question)

    pairs = [[question, node.node.get_content()] for node in candidates]
    scores = reranker.predict(pairs)
    for node, score in zip(candidates, scores):
        node.score = float(score)
    candidates.sort(key=lambda x: x.score, reverse=True)

    context = "\n\n---\n\n".join(node.node.get_content() for node in candidates[:4])
    prompt = POLICY_QA_PROMPT.format(context=context, question=question)
    return call_llm(prompt)

print(await policy_rag_agent("What's the company 401k match?"))


## 9. Eval loop (kept from your notebook, deduplicated to one clean version)

In [ ]:
eval_questions = {
    "Q6": "How do I request PTO?",
    "Q11": "Who's on-call for engineering the week of July 15? And how do I reach them if there's a P1 incident?",
    "Q15": "If I'm on probation and I get sick for a week, do I have PTO to cover it?",
    "Q21": "I'm a Solutions Engineer. What's my sales quota?",
    "Q26": "What's the company's stock ticker symbol?",
    "Q28": "How do I reset my Okta password?",
}

for qid, q in eval_questions.items():
    print(f"\n{'='*60}\n{qid}: {q}\n{'='*60}")
    print(await policy_rag_agent(q))


## 10. Multi-agent state (Stage A — next step picks up here)

In [ ]:
from typing import TypedDict, List, Annotated
import operator

class AgentState(TypedDict):
    task: str
    route: List[str]
    outputs: Annotated[List[dict], operator.add]
    final_output: str


## What was removed as duplicate/dead code
- Multiple repeated `ollama serve` / `nest_asyncio.apply()` calls scattered across cells — consolidated into Cells 1-2.
- Two abandoned reranker attempts using `FlagEmbedding` (tokenizer version incompatibility, monkey-patch attempts) — removed entirely; `CrossEncoder` was the final working choice.
- Multiple redundant `POLICY_QA_PROMPT` definitions (4 versions existed) — kept only the final, most complete one.
- Repeated `ChatOllama` initialization — no longer needed at all, since generation now goes through OpenRouter.
- Duplicate eval-question dictionaries (2 near-identical versions existed) — merged into one.
- Debug-only one-off cells (checking chunk counts, checking for "Fidelity"/"Priya Nair" substrings) — kept the *pattern* as reusable, didn't keep every ad-hoc instance.

## Still true / worth remembering
- BM25 + local reranking stay local — no reason to move them to an API, they're not the bottleneck.
- Only the generation step needed OpenRouter, since that's where you wanted OpenRouter's model quality/reliability over the local 7B.
- Next step: Router node + 3 more sub-agents (Researcher, Marketing, Writer/Editor), all calling `call_llm()`.
